# ColabUMA Workflow — 最適化 → スキャン → 遷移状態を1つのセッションで

[ColabUMA_Opt](https://colab.research.google.com/github/RChibaChiba/ColabUMA/blob/main/ColabUMA_Opt.ipynb)（構造最適化）、[ColabUMA_Scan](https://colab.research.google.com/github/RChibaChiba/ColabUMA/blob/main/ColabUMA_Scan.ipynb)（緩和スキャン）、[ColabUMA_TS](https://colab.research.google.com/github/RChibaChiba/ColabUMA/blob/main/ColabUMA_TS.ipynb)（遷移状態探索）を、**同じセッションのまま続けて** 実行するノートブックです。Colab ではノートごとにセッションが分かれるので、別々のノートで続けるとそのたびにインストールとモデル読み込みをやり直すことになります。このノートなら1回で済み、前の段階の構造をそのまま次の段階に渡します。

計算の中身は3つのノートと同じ共通コード（`colabuma` パッケージ）を使っています。

## 流れ

1. **構造最適化**：入力構造を最適化する（反応物）
2. **スキャン**：最適化した構造から、指定した結合長・角度・二面角を動かす。スキャンの後に、最後の点からさかのぼった極小を最適化する（生成物）
3. **遷移状態**：反応物と生成物から DMF で TS の初期構造を作り、Sella で最適化する。「スキャンの最高点から Sella」を選ぶと、DMF を飛ばしてスキャンの最高点から始める
4. **DFT 一点計算**：反応物・TS・生成物のエネルギーを DFT で計算する

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 初回のみ、左サイドバーの🔑（シークレット）に `HF_TOKEN` を登録し、「ノートブックからのアクセス」をオンにする
3. **1. 入力フォーム** を埋めて **ランタイム → すべて実行** を押し、**2.** で入力構造（xyz または gjf）を選ぶ。あとは放置できます
4. 最後のセルで、すべての段階の結果を1つの ZIP にまとめてダウンロードします

途中の段階だけやり直したいときは、フォームを直してから、そのセル以降を選んで実行してください（**ランタイム → 以降のセルを実行**）。インストールとモデル読み込みはスキップされます。

原子番号が分からないときは、一度実行すると「5. 構造の準備」に番号付きの3D表示が出ます。

> **事前準備**：モデルの重み [facebook/UMA](https://huggingface.co/facebook/UMA) はアクセス申請が必要です。Hugging Faceにログインしてページ上でFAIR Chemistry License v1に同意し、承認後に [Access Tokens](https://huggingface.co/settings/tokens) で Read 権限のトークンを作成してください。

## 注意事項

- スキャンで結合を作る・切る方向に十分動かし、スキャン後の最適化が生成物に落ちるようにしてください。スキャン後の最適化構造が入力と同じなら、DMF は始められません。
- 各段階の注意事項は、それぞれのノートの説明を見てください。

## ライセンス

- このノートと共通コード（`colabuma`）：[MIT License](https://github.com/RChibaChiba/ColabUMA/blob/main/LICENSE)
- UMAのモデル重み：[FAIR Chemistry License v1](https://huggingface.co/facebook/UMA) に従います
- 実行時にインストールするライブラリは、それぞれのライセンスに従います。一覧は [README のライセンス](https://github.com/RChibaChiba/ColabUMA#ライセンス) にあります

## 引用

論文で使う場合は、以下を引用してください。

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

遷移状態探索（DMF・Sella）を使った場合は、次も引用してください。

- S.-i. Koda and S. Saito, *J. Open Source Softw.* **11**, 10379 (2026).（PyDMF）
- S.-i. Koda and S. Saito, *J. Chem. Theory Comput.* **20**, 2798–2811 (2024).（Direct MaxFlux）
- S.-i. Koda and S. Saito, *J. Chem. Theory Comput.* **20**, 7176–7187 (2024); **21**, 3513–3522 (2025).（FB-ENM、CFB-ENM）
- E. D. Hermes, K. Sargsyan, H. N. Najm, J. Zádor, *J. Chem. Theory Comput.* **18**, 6974–6988 (2022).（Sella）
- DFT の一点計算を使った場合：PySCF（Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020)）と GPU4PySCF（X. Wu *et al.*, arXiv:2404.09452 (2024)）

In [ ]:
#@title 1. 入力フォーム { display-mode: "form" }
#@markdown 入力が終わったら **ランタイム → すべて実行** を押してください。次のセルで xyz または gjf ファイルを選ぶと、あとは放置できます。

#@markdown ### 出力
output_name = "result"  #@param {type:"string"}

#@markdown ### 構造
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### 1. 構造最適化
run_vibrations = False  #@param {type:"boolean"}

#@markdown ### 2. スキャン
#@markdown 原子番号（1始まり）。2個なら結合長、3個なら角度、4個なら二面角です。開始値が空欄なら最適化後の値から始めます。
scan_atoms = "1 2"  #@param {type:"string"}
scan_start = ""  #@param {type:"string"}
scan_n_steps = 20  #@param {type:"integer"}
scan_step_size = -0.1  #@param {type:"number"}

#@markdown ### 3. 遷移状態
ts_source = "DMF → Sella（最適化構造とスキャン後の最適化構造から）"  #@param ["DMF → Sella（最適化構造とスキャン後の最適化構造から）", "スキャンの最高点から Sella"]

#@markdown DMF の設定（DMF を選んだとき）
#@markdown 経路の途中でエネルギーを計算する点の数。多いほど TS の初期構造が正確になり、時間もかかります。
dmf_nmove = 5  #@param {type:"integer"}
dmf_convergence = "middle"  #@param ["loose", "middle", "tight"]
dmf_max_iter = 300  #@param {type:"integer"}

#@markdown 最初のヘシアンを UMA の数値微分で作ります（力の計算 6N 回）。オフにすると近似ヘシアンで始めます。
ts_exact_hessian = True  #@param {type:"boolean"}
#@markdown TS から虚振動の向きに少しずらして両側に最適化し、どの構造につながるか確かめます（簡易IRC）。
check_connection = True  #@param {type:"boolean"}

#@markdown ### 4. DFT 一点計算（gpu4pyscf）
#@markdown UMA の構造で DFT のエネルギーを計算し、活性化エネルギーを比べます。ωB97M-V/def2-TZVPD が UMA（OMol25）の学習データと同じ計算レベルです。def2-TZVPD は T4 では1点に数十分かかることがあります。
run_dft = True  #@param {type:"boolean"}
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V（OMol25と同じ）", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]
#@markdown ZIP に入れる Gaussian の TS 計算用 gjf のルート行。
gaussian_route = "# opt=(ts,calcfc,noeigentest) freq b3lyp/def2svp empiricaldispersion=gd3bj"  #@param {type:"string"}

#@markdown ### 計算条件
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown 収束判定 fmax（eV/Å）。normal は Gaussian の既定、tight は Gaussian の opt=tight と同じくらいです。
convergence = "normal（0.02）"  #@param ["loose（0.05）", "normal（0.02）", "tight（0.001）", "数値指定"]
#@markdown 「数値指定」を選んだときの fmax（eV/Å）。
fmax_custom = 0.02  #@param {type:"number"}
#@markdown 収束判定と最大ステップ数は、すべての段階で共通です（スキャンでは1点あたり）。
max_steps = 500  #@param {type:"integer"}

In [ ]:
#@title 2. ファイル選択と事前確認 { display-mode: "form" }
#@markdown ファイルの選択とトークンの確認をここで済ませます。このあとのインストールと計算は放置できます。

import os
import subprocess
import sys

COLABUMA_REF = "main"  # 共通コード（colabuma パッケージ）を取ってくる GitHub のブランチ


def _use_colabuma(ref):
    # ノートと同じフォルダに colabuma があればそれを、なければ GitHub から取ってきて使う
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabUMA" if os.path.isdir("/content") else os.path.expanduser("~/ColabUMA")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabUMA.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"ColabUMA の共通コードを GitHub から取得できませんでした（ブランチ {ref}）。\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, gaussian, opt, output, scan, structure, ts, ui
from colabuma.ui import ColabUMAError, heading, ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
spec = scan.parse_spec(scan_atoms, scan_start, scan_n_steps, scan_step_size, opt_before=False, opt_after=True)
use_dmf = ts_source.startswith("DMF")
if dmf_nmove < 1 or dmf_max_iter < 1:
    raise ColabUMAError("DMF の nmove と最大反復回数は1以上にしてください。", "通常は nmove = 5、最大反復回数 = 300 程度です。")
if run_dft:
    dft.parse_method(dft_functional, dft_basis)
env.check_gpu()
env.ensure_token()
upload_name, upload_data = structure.upload()
scan.check_atom_range(spec, structure.quick_atom_count(upload_name, upload_data),
                      "フォームを直して、もう一度すべて実行してください。ファイル選択は Cancel upload で前回のファイルを使えます。")
ok("入力の確認が終わりました。ここからは放置できます。")

In [ ]:
#@title 3. セットアップ（インストール・認証） { display-mode: "form" }
DEVICE = env.setup(["ts", "dft"] if run_dft else ["ts"])

In [ ]:
#@title 4. モデル読み込み { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. 構造の準備 { display-mode: "form" }
atoms0, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity)
ok(f"{atoms0.get_chemical_formula()}（{len(atoms0)} 原子）の初期構造を用意しました。")
sc = scan.prepare(spec, atoms0)
all_files, all_logs = {}, []

In [ ]:
#@title 6. 構造最適化 { display-mode: "form" }
heading("1. 構造最適化")
print(f"{optimizer_name} で最適化します（fmax < {fmax} eV/Å、最大 {max_steps} ステップ）")
best = opt.optimize(atoms0, calc, job, "構造")
vib_result = opt.vibrations(best["atoms"], calc) if run_vibrations else None
_f, _l = opt.report(best, vib_result, job)
all_files.update(_f)
all_logs.append(_l)

In [ ]:
#@title 7. スキャン { display-mode: "form" }
heading("2. スキャン")
scan_res = scan.run(sc, best["atoms"], calc, job, skip_preopt_result=best)
_f, _l = scan.report(scan_res, job, atoms0)
for _k in ("preopt.xyz", "preopt.log"):  # 構造最適化の結果（opt.xyz / opt.log）と同じなので入れない
    _f.pop(_k, None)
all_files.update(_f)
all_logs.append(_l)

In [ ]:
#@title 8. 遷移状態 { display-mode: "form" }
heading("3. 遷移状態")
reactant = best["atoms"]
product = ts.check_endpoints(reactant, scan_res["post"]["atoms"])
endpoints = {"reactant": {"atoms": reactant, "energy": best["energy"], "converged": best["converged"]},
             "product": {"atoms": product, "energy": scan_res["post"]["energy"], "converged": scan_res["post"]["converged"]}}
dmf_res = None
if use_dmf:
    dmf_res = ts.run_dmf(reactant, product, calc, job, nmove=dmf_nmove, tol=dmf_convergence, max_iter=dmf_max_iter)
    ts_guess = dmf_res["ts_guess"]
else:
    ts_guess = scan_res["highest"]["atoms"]
    print(f"スキャンの最高点（点 {scan_res['highest']['point']}）から Sella で最適化します。")
ts_res = ts.run_sella(ts_guess, calc, job, exact_hessian=ts_exact_hessian)
ts_vib = ts.check_ts(ts_res, calc)
conn = None
if check_connection and ts_vib is not None and ts_vib["n_imag"] >= 1:
    conn = ts.connect(ts_res["atoms"], ts_vib, calc, job, reactant, product)
_f, ts_log, uma_rows = ts.report(job, reactant, product, dmf_res, ts_res, ts_vib, conn, endpoints)
_f["ts.gjf"] = gaussian.gjf(ts_res["atoms"], job, gaussian_route, f"{job.out_prefix} TS from ColabUMA ({model_name})",
                              chk=f"{job.out_prefix}_ts.chk")
all_files.update(_f)
all_logs.append([f"[遷移状態] TS の初期構造: {ts_source}"] + ts_log)

In [ ]:
#@title 9. DFT 一点計算 { display-mode: "form" }
heading("4. DFT 一点計算")
if run_dft:
    dft_res = dft.run(ts.dft_points(ts_res, endpoints, conn), charge, multiplicity, dft_functional, dft_basis)
    _f, _l = dft.report(dft_res, uma_rows)
    all_files.update(_f)
    all_logs.append(_l)
else:
    print("DFT 一点計算はオフです。")

In [ ]:
#@title 10. 保存 { display-mode: "form" }
all_files["log.txt"] = output.log_text("ColabUMA Workflow 計算ログ", source_desc, job, all_logs)
zip_path = output.save_zip(job.out_prefix, all_files)